# Synthetic multi-semester planning comparison
All rules, records and offerings are invented. This is an algorithm experiment, not a graduation prediction. See docs/PLANNING_METHODOLOGY.md. Timing repetitions are not independent student samples. Controls and designed traps must be reported separately.


In [1]:
import csv, hashlib, json, statistics
from pathlib import Path
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'DegreePilot.slnx').exists())
folder = root / 'benchmarks/degree_planning'
manifest = json.loads((folder / 'manifest.json').read_text())
rows = list(csv.DictReader((folder / 'results.csv').open(newline='')))
assert len(rows) == manifest['cases'] * manifest['repetitions'] * 2
print('Cases:', manifest['cases'], 'runs:', len(rows), 'Python:', manifest['python'], 'OR-Tools:', manifest['ortools'])
print('Synthetic only:', manifest['synthetic_only'], 'dataset SHA256:', manifest['dataset_sha256'])


Cases: 22 runs: 132 Python: 3.12.10 OR-Tools: 9.15.6755
Synthetic only: True dataset SHA256: 8bfe27db89c1e9dc183f9d1b3ac1389fbe1288c80b5819c54d5ff4afdcf17c43


In [2]:
for algorithm in ('cp_sat', 'greedy'):
    own = [r for r in rows if r['algorithm'] == algorithm]
    counts = {s: sum(r['status'] == s for r in own) for s in sorted({r['status'] for r in own})}
    print(algorithm, counts, 'median ms:', statistics.median(float(r['elapsed_ms']) for r in own))
for family in sorted({r['family'] for r in rows}):
    print(family, {a: sum(r['family'] == family and r['algorithm'] == a and r['status'] == 'FEASIBLE' for r in rows) for a in ('cp_sat', 'greedy')})
for case_id in ('deadline-trap-3', 'deadline-trap-4', 'approved-P-repeat'):
    print(case_id, [(r['algorithm'], r['status'], r['completion_terms'], r['attempted_credits']) for r in rows if r['case_id'] == case_id and r['repetition'] == '0'])


cp_sat {'FEASIBLE': 42, 'INFEASIBLE': 21, 'TIMEOUT': 3} median ms: 27.840049999999998
greedy {'FEASIBLE': 27, 'NO_SOLUTION': 39} median ms: 1.6622
complete-control {'cp_sat': 3, 'greedy': 3}
designed-trap {'cp_sat': 6, 'greedy': 3}
fixture {'cp_sat': 30, 'greedy': 15}
infeasible-control {'cp_sat': 0, 'greedy': 0}
repeat-control {'cp_sat': 3, 'greedy': 3}
timeout-control {'cp_sat': 0, 'greedy': 3}
deadline-trap-3 [('cp_sat', 'FEASIBLE', '3', '9'), ('greedy', 'NO_SOLUTION', '', '0')]
deadline-trap-4 [('cp_sat', 'FEASIBLE', '3', '9'), ('greedy', 'FEASIBLE', '4', '9')]
approved-P-repeat [('cp_sat', 'FEASIBLE', '2', '6'), ('greedy', 'FEASIBLE', '2', '6')]


In [3]:
cases = json.loads((folder / 'cases.json').read_text())
digest = lambda value: hashlib.sha256(json.dumps(value, sort_keys=True, separators=(',', ':')).encode()).hexdigest()
assert digest(cases) == manifest['dataset_sha256']
by_id = {c['id']: c['request'] for c in cases}
assert all(digest(by_id[r['case_id']]) == r['input_sha256'] for r in rows)
assert all(int(r['invariant_violations']) == 0 for r in rows)
joint = []
for case_id in by_id:
    pair = {r['algorithm']: r for r in rows if r['case_id'] == case_id and r['repetition'] == '0'}
    if all(pair[a]['status'] == 'FEASIBLE' for a in ('cp_sat', 'greedy')):
        joint.append(int(pair['greedy']['completion_terms']) - int(pair['cp_sat']['completion_terms']))
print('Paired feasible cases:', len(joint), 'CP completion-term improvements:', joint)
print('All recorded invariants and input hashes verified. Independent Python and ASP.NET replay tests are required separately.')


Paired feasible cases: 8 CP completion-term improvements: [0, 0, 0, 0, 0, 1, 0, 0]
All recorded invariants and input hashes verified. Independent Python and ASP.NET replay tests are required separately.
